# LanceDB - a vector database for LLM applications

In [1]:
import lancedb

db = lancedb.connect(uri = "vector_database")
db


LanceDBConnection(uri='c:\\Users\\ricka\\Documents\\Course\\ai-ops-course\\09_lancedb\\vector_database')

In [2]:
db.uri

'c:\\Users\\ricka\\Documents\\Course\\ai-ops-course\\09_lancedb\\vector_database'

## Create a table

In [3]:
import json

with open("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [4]:
table_animals = db.create_table("animals_text", exist_ok=True, data = data)
table_animals

LanceTable(name='animals_text', _conn=LanceDBConnection(uri='c:\\Users\\ricka\\Documents\\Course\\ai-ops-course\\09_lancedb\\vector_database'))

In [5]:
table_animals = db.create_table("animals_text", data = data, mode="overwrite")
table_animals

LanceTable(name='animals_text', _conn=LanceDBConnection(uri='c:\\Users\\ricka\\Documents\\Course\\ai-ops-course\\09_lancedb\\vector_database'))

In [6]:
table_animals.head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

In [7]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


In [8]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

table_animals.add(more_data)

AddResult(version=3)

In [9]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create an empty table and then delete it

In [10]:
from lancedb.pydantic import LanceModel, Vector

class JokeSchema(LanceModel):
    joke: str
    rating: int

db.create_table(name="jokes", schema=JokeSchema, exist_ok=True)
db

LanceDBConnection(uri='c:\\Users\\ricka\\Documents\\Course\\ai-ops-course\\09_lancedb\\vector_database')

In [11]:
db.table_names()

C:\Users\ricka\AppData\Local\Temp\ipykernel_16668\763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text', 'jokes', 'jokes_nvidia']

In [12]:
db.drop_table("jokes")

In [13]:
db.table_names()

C:\Users\ricka\AppData\Local\Temp\ipykernel_16668\763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text', 'jokes_nvidia']

## Open existing table

In [14]:
db.open_table("animals_text").head(2)

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1]]]

## Vector search in LanceDb

In [15]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [16]:
query_vector = [0.5, 0.2, 0.9]
table_animals.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.2673


## Embedding API
- Idea: want to put in text -> and it will automatically generate vector embeddings
- put in a query adn it will automatically generate vector embeddings
- calculate closest distances

In [17]:
import lancedb
import pandas as pd

df = pd.read_json("data/jokes.json")
db = lancedb.connect("vector_database")
df.head()

,jokes
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [18]:
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from custom_embeddings.nvidia_nemotron import NvidiaNemotronEmbedding

model = get_registry().get("sentence-transformers").create(name="intfloat/multilingual-e5-small", device="cpu")

class JokeModel(LanceModel):
    joke: str = model.SourceField() # marks text as input to embedding function
    vector: Vector(model.ndims()) = model.VectorField() # vector is destination of computed embedding

table_jokes = db.create_table("jokes", schema = JokeModel, exist_ok=True)
table_jokes

c:\Users\ricka\Documents\Course\ai-ops-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ricka\Documents\Course\ai-ops-course\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ricka\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an

LanceTable(name='jokes', _conn=LanceDBConnection(uri='c:\\Users\\ricka\\Documents\\Course\\ai-ops-course\\09_lancedb\\vector_database'))

In [19]:
import pandas as pd 
with open("data/jokes.json", "r", encoding="utf-8") as file:
    jokes_data = json.loads(file.read())

df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [20]:
df_jokes["joke"].iloc[0]

'Parallel lines have so much in common—it’s sad they’ll never meet.'

In [21]:
table_jokes.head()

pyarrow.Table
joke: string not null
vector: fixed_size_list<item: float>[384]
  child 0, item: float
----
joke: []
vector: []

In [22]:
table_jokes.add(df_jokes)

AddResult(version=2)

In [23]:
table_jokes.head()

pyarrow.Table
joke: string not null
vector: fixed_size_list<item: float>[384]
  child 0, item: float
----
joke: [["Parallel lines have so much in common—it’s sad they’ll never meet.","ETL stands for “Extract, Transform, Leave for the next person.”","What do you call a snake that runs your scripts? A py-thon (ton of errors).","Gold walks into a bar. The bartender says, “Au, get out!”","C# devs don’t argue; they just throw exceptions."]]
vector: [[[0.037198022,-0.01991781,-0.051841024,-0.032692872,0.104856074,...,-0.07006689,0.04333404,0.047141075,0.067228846,0.055702467],[0.05049738,-0.0064256503,-0.07101135,-0.027726112,0.04461316,...,-0.015193849,-0.007357265,0.03242999,0.027537324,0.07036028],[0.073168345,-0.004549964,-0.027777124,-0.054147594,0.044475716,...,-0.054630622,0.04199846,0.05612801,0.05367641,0.045334365],[0.045366433,-0.05297,-0.06690079,-0.028929865,0.11357268,...,-0.060623825,0.084598385,0.050644673,0.06691325,0.011901182],[0.028011477,0.004646022,-0.057663728,-0.06480

## Perform vector search

In [24]:
table_jokes.search("data engineering jokes").limit(5).to_pandas()

,joke,vector,_distance
0,Why do data engineers hate nature? Too many un...,"[0.057509, 0.012957, -0.081348, -0.042842, 0.0...",0.263454
1,"Data engineer motto: If it works, don’t touch ...","[0.003134, -0.036774, -0.050577, -0.060755, 0....",0.275054
2,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.309797
3,"Our data pipeline broke… but don't worry, it w...","[0.032507, -0.009452, -0.025917, -0.060267, 0....",0.314785
4,I asked the data lake if it had my file. It sa...,"[0.069376, -0.0138, -0.052487, -0.050929, 0.08...",0.330099


In [25]:
table_jokes.search("give me som jokes on C#").limit(5).to_pandas()

,joke,vector,_distance
0,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.257035
1,Why is C# like a musical? It has so many classes.,"[0.021061, -0.003477, -0.029677, -0.060304, 0....",0.273192
2,C# devs don’t argue; they just throw exceptions.,"[0.028011, 0.004646, -0.057664, -0.064801, 0.0...",0.287758
3,C# programmers love coffee—they're always work...,"[-0.011444, -0.021634, -0.055446, -0.035535, 0...",0.307085
4,Why did the C# developer go broke? He kept usi...,"[0.020794, 0.007793, -0.040906, -0.02401, 0.03...",0.316894


In [26]:
table_jokes.search("give me nature related jokes").limit(5).to_pandas()

,joke,vector,_distance
0,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.323299
1,Why do data engineers hate nature? Too many un...,"[0.057509, 0.012957, -0.081348, -0.042842, 0.0...",0.327534
2,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.335890
3,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.355142
4,I asked the data lake if it had my file. It sa...,"[0.069376, -0.0138, -0.052487, -0.050929, 0.08...",0.364861


## Hybrid search

hybrid search combines traditional keyword-based search with vector-based search to provide more relevant results. This approach leverages the strengths of both methods, allowing for more accurate and context-aware retrieval of information.

In [27]:
# to enable keyword based search on joke column
table_jokes.create_fts_index("joke", replace=True)

C:\Users\ricka\AppData\Local\Temp\ipykernel_16668\878140144.py:2: DeprecatedWarning: create_fts_index is deprecated as of 0.25.0. Use create_index() with config=FTS() instead.
  table_jokes.create_fts_index("joke", replace=True)


In [28]:
from lancedb import rerankers

reranker = rerankers.RRFReranker()

results = table_jokes.search(
    "give me nature related jokes",
    query_type="hybrid",
    vector_column_name="vector",
    fts_columns="joke"
).rerank(reranker=reranker).limit(5).to_pandas()

results

,joke,vector,_relevance_score
0,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.032787
1,Why do data engineers hate nature? Too many un...,"[0.057509, 0.012957, -0.081348, -0.042842, 0.0...",0.032258
2,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.015873
3,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.015625
4,I asked the data lake if it had my file. It sa...,"[0.069376, -0.0138, -0.052487, -0.050929, 0.08...",0.015385


## Rule of thumb

- For exact matching -> FTS
- For meaning based matching -> Vector search
- Unpredictable or mixed queries -> Hybrid search

In [33]:
from lancedb.pydantic import Vector
from lancedb.embeddings import get_registry
from dotenv import load_dotenv

load_dotenv()

model = get_registry().get("cohere").create(name="embed-multilingual-v3.0")

class JokeModel(LanceModel):
    jokes: str = model.SourceField()
    vector: Vector(model.ndims()) = model.VectorField()

print(f"{model.ndims() = }")
db.create_table("jokes", schema=JokeModel, exist_ok=True, mode="overwrite")

model.ndims() = 1024


LanceTable(name='jokes', _conn=LanceDBConnection(uri='c:\\Users\\ricka\\Documents\\Course\\ai-ops-course\\09_lancedb\\vector_database'))

In [36]:
pd.read_json("data/jokes.json")
df.head(2)

,jokes
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."


In [39]:
db["jokes"].add(df)

AddResult(version=5)

In [40]:
db["jokes"].to_pandas().head()

,jokes,vector
0,Parallel lines have so much in common—it’s sad...,"[-0.00301, 0.001863, -0.014427, 0.051575, -0.0..."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.01239, 0.0047, -0.091431, 0.080627, -0.0505..."
2,What do you call a snake that runs your script...,"[0.019012, 0.044525, -0.021454, 0.02887, -0.00..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.019958, 0.060089, 0.012733, 0.024048, -0.07..."
4,C# devs don’t argue; they just throw exceptions.,"[0.013275, 0.002462, -0.057007, 0.060883, -0.0..."


In [42]:
db["jokes"].search("ge mig ett python skämt").limit(3).to_pandas()

,jokes,vector,_distance
0,What do you call a snake that runs your script...,"[0.019012, 0.044525, -0.021454, 0.02887, -0.00...",0.813401
1,Why did the Python programmer get bitten? Beca...,"[0.012062, 0.057861, -0.028549, 0.01281, -0.00...",0.936815
2,Python is so friendly even whitespace gets a say.,"[0.014603, 0.016724, -0.002628, 0.038391, -0.0...",1.055928


In [48]:
# keyword based matching on jokes column
from lancedb.index import FTS
db["jokes"].create_index("jokes", config=FTS(), replace=True)

In [56]:
db["jokes"].search(
    "give me math jokes", 
    query_type="hybrid", 
    vector_column_name="vector", 
    fts_columns="jokes",).limit(5).to_pandas()


,jokes,vector,_relevance_score
0,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",0.032258
1,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",0.031778
2,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",0.016393
3,The C# compiler walked into a bar. The bartend...,"[0.003672, 0.006752, -0.076904, 0.026688, -0.0...",0.015873
4,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",0.015625


In [55]:
db["jokes"].search(
    "give me math jokes", 
    query_type="fts", 
    vector_column_name="vector", 
    fts_columns="jokes",).limit(5).to_pandas()

,jokes,vector,_score
0,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",3.530949
1,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",3.324148


In [57]:
from lancedb import rerankers

reranker = rerankers.RRFReranker()

db["jokes"].search(
    "give me math jokes", 
    query_type="hybrid", 
    vector_column_name="vector", 
    fts_columns="jokes",).rerank(reranker).limit(5).to_pandas()

,jokes,vector,_relevance_score
0,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",0.032258
1,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",0.031778
2,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",0.016393
3,The C# compiler walked into a bar. The bartend...,"[0.003672, 0.006752, -0.076904, 0.026688, -0.0...",0.015873
4,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",0.015625
